# Lab3 - Assignment Sentiment

Copyright: Vrije Universiteit Amsterdam, Faculty of Humanities, CLTL

This notebook describes the LAB-3 assignment of the Text Mining course. It is about sentiment analysis.

The aims of the assignment are:
* Learn how to run a rule-based sentiment analysis module (VADER)
* Learn how to run a machine learning sentiment analysis module (Scikit-Learn/ Naive Bayes)
* Learn how to run scikit-learn metrics for the quantitative evaluation
* Learn how to perform and interpret a quantitative evaluation of the outcomes of the tools (in terms of Precision, Recall, and F<sub>1</sub>)
* Learn how to evaluate the results qualitatively (by examining the data) 
* Get insight into differences between the two applied methods
* Get insight into the effects of using linguistic preprocessing
* Be able to describe differences between the two methods in terms of their results
* Get insight into issues when applying these methods across different  domains

In this assignment, you are going to create your own gold standard set from 50 tweets. You will the VADER and scikit-learn classifiers to these tweets and evaluate the results by using evaluation metrics and inspecting the data.

We recommend you go through the notebooks in the following order:
* **Read the assignment (see below)**
* **Lab3.2-Sentiment-analysis-with-VADER.ipynb**
* **Lab3.3-Sentiment-analysis.with-scikit-learn.ipynb**
* **Answer the questions of the assignment (see below) using the provided notebooks and submit**

In this assignment you are asked to perform both quantitative evaluations and error analyses:
* a quantitative evaluation concerns the scores (Precision, Recall, and F<sub>1</sub>) provided by scikit's classification_report. It includes the scores per category, as well as micro and macro averages. Discuss whether the scores are balanced or not between the different categories (positive, negative, neutral) and between precision and recall. Discuss the shortcomings (if any) of the classifier based on these scores
* an error analysis regarding the misclassifications of the classifier. It involves going through the texts and trying to understand what has gone wrong. It servers to get insight in what could be done to improve the performance of the classifier. Do you observe patterns in misclassifications?  Discuss why these errors are made and propose ways to solve them.

## Credits
The notebooks in this block have been originally created by [Marten Postma](https://martenpostma.github.io) and [Isa Maks](https://research.vu.nl/en/persons/e-maks). Adaptations were made by [Filip Ilievski](http://ilievski.nl).

## Part I: VADER assignments


### Preparation (nothing to submit):
To be able to answer the VADER questions you need to know how the tool works. 
* Read more about the VADER tool in [this blog](https://www.geeksforgeeks.org/python-sentiment-analysis-using-vader/).  
* VADER provides 4 scores (positive, negative, neutral, compound). Be sure to understand what they mean and how they are calculated.
* VADER uses rules to handle linguistic phenomena such as negation and intensification. Be sure to understand which rules are used, how they work, and why they are important.
* VADER makes use of a sentiment lexicon. Have a look at the lexicon. Be sure to understand which information can be found there (lemma?, wordform?, part-of-speech?, polarity value?, word meaning?) What do all scores mean? https://github.com/cjhutto/vaderSentiment/blob/master/vaderSentiment/vader_lexicon.txt) 


### [3.5 points] Question1:

Regard the following sentences and their output as given by VADER. Regard sentences 1 to 7, and explain the outcome **for each sentence**. Take into account both the rules applied by VADER and the lexicon that is used. You will find that some of the results are reasonable, but others are not. Explain what is going wrong or not when correct and incorrect results are produced. 

```
INPUT SENTENCE 1 I love apples
VADER OUTPUT {'neg': 0.0, 'neu': 0.192, 'pos': 0.808, 'compound': 0.6369}

INPUT SENTENCE 2 I don't love apples
VADER OUTPUT {'neg': 0.627, 'neu': 0.373, 'pos': 0.0, 'compound': -0.5216}

INPUT SENTENCE 3 I love apples :-)
VADER OUTPUT {'neg': 0.0, 'neu': 0.133, 'pos': 0.867, 'compound': 0.7579}

INPUT SENTENCE 4 These houses are ruins
VADER OUTPUT {'neg': 0.492, 'neu': 0.508, 'pos': 0.0, 'compound': -0.4404}

INPUT SENTENCE 5 These houses are certainly not considered ruins
VADER OUTPUT {'neg': 0.0, 'neu': 0.51, 'pos': 0.49, 'compound': 0.5867}

INPUT SENTENCE 6 He lies in the chair in the garden
VADER OUTPUT {'neg': 0.286, 'neu': 0.714, 'pos': 0.0, 'compound': -0.4215}

INPUT SENTENCE 7 This house is like any house
VADER OUTPUT {'neg': 0.0, 'neu': 0.667, 'pos': 0.333, 'compound': 0.3612}
```

Sentence 1: 
"love" is in the VADER lexicon with a high positive score (~3.0). "apples" and "I" are neutral, so teh result is reasonable.

Sentence 2: 
Looks correct, since VADER applies a negation rule where negators like "don't" within a 3 token window before a sentiment word flip its polarity. So "don't" negates "love", turning the positive score negative.

Sentence 3: 
Also looks correct, same as sentence 1 (but rather the opposite of it because there is no word "don't" - no negator), and also the emoticon/emoji ":-)" exists in the VADER lexicon with its own positive score. So VADER's emoticon rule adds this on top of "love", which boosts the compound score (increases it even more).

Sentence 4:
This is incorrect. The word "ruins" (plural noun, which means old destroyed buildings) has a negative entry in the VADER lexicon. But here it is just used as a neutral descriptive noun (saying houses are ruins is a factual/architectural observation) not an expression of negative sentiment. But VADER can't distinguish this noun sense (old buildings) from a verb sense (to ruin/destroy), so it wrongly scores this as negative.

Sentence 5:
This one could be seen as questionable, but it seems rather correct. Here VADER's negation rule kicks in with: "not" negates "ruins", which  flipps its negative score to positive., then there is also "certainly", which is a degree modifier (booster) in the VADER lexicon, which amplifies the result. While the logic is internally consistent, we think the sentence is essentially neutral, it just says the houses are not in bad condition, it could be said that describing the hosues as "certainly not considered ruins" could be seen as a positive rem ark about teh houses, but the sentence does not asisgn any clear positive words about the hosues so it ends up sounding rather neutral. So scoring it as noticeably positive (0.5867) seems like a bit of an overestimation.

Sentence 6:
This is incorrect, teh sentence is not negative, it is just neutral. The word "lies" in the VADER lexicon refers to telling lies (dishonesty), which is negative. But actually here "lies" means "to lay down", a completely neutral positional verb. VADER has no word sense disambiguation, it cannot tell these apart and wrongly treats this neutral sentence as negative.

Setnence 7:
This also seems incorrect. What happens here is that: "like" in the VADER lexicon is scored as a positive word (to like something = enjoyment). But in this sentence "like" is a preposition/conjunction meaning "similar to" "alike", which carries no sentiment, its just a neutral comparison word. Again VADER cannot distinguish word senses, so a fully neutral comparative sentence gets scored as positive, in thsi case, not being  correct.

### [Points: 2.5] Exercise 2: Collecting 50 tweets for evaluation
Collect 50 tweets. Try to find tweets that are interesting for sentiment analysis, e.g., very positive, neutral, and negative tweets. These could be your own tweets (typed in) or collected from the Twitter stream. If you have trouble accessing Twitter, try to find an existing dataset (on websites like kaggle or huggingface).

We will store the tweets in the file **my_tweets.json** (use a text editor to edit).
For each tweet, you should insert:
* sentiment analysis label: negative | neutral | positive (this you determine yourself, this is not done by a computer)
* the text of the tweet
* the Tweet-URL

from:
```
    "1": {
        "sentiment_label": "",
        "text_of_tweet": "",
        "tweet_url": "",
```
to:
```
"1": {
        "sentiment_label": "positive",
        "text_of_tweet": "All across America people chose to get involved, get engaged and stand up. Each of us can make a difference, and all of us ought to try. So go keep changing the world in 2018.",
        "tweet_url" : "https://twitter.com/BarackObama/status/946775615893655552",
    },
```

You can load your tweets with human annotation in the following way.

In [20]:
import json

In [21]:
my_tweets = json.load(open('my_tweets.json'))

In [22]:
for id_, tweet_info in my_tweets.items():
    print(id_, tweet_info)
    break

# Please find the script we used to get the tweets in the file `tweet_extraction_huggingface.py` in this directory
# We got teh tweets dataset from Hugging Face

1 {'sentiment_label': 'positive', 'text_of_tweet': 'So Conor McGregor is ALREADY the favorite leading up to the fight in December. I feel the odds will fluctuate quite a bit! #UFC #UFC194', 'tweet_url': 'https://huggingface.co/datasets/tweet_eval'}


### [5 points] Question 3:

Run VADER on your own tweets (see function **run_vader** from notebook **Lab2-Sentiment-analysis-using-VADER.ipynb**). You can use the code snippet below this explanation as a starting point. 
* [2.5 points] a. Perform a quantitative evaluation. Explain the different scores, and explain which scores are most relevant and why.
* [2.5 points] b. Perform an error analysis: select 10 positive, 10 negative and 10 neutral tweets that are not correctly classified and try to understand why. Refer to the VADER-rules and the VADER-lexicon. Of course, if there are less than 10 errors for a category, you only have to check those. For example, if there are only 5 errors for positive tweets, you just describe those.

In [23]:
def vader_output_to_label(vader_output):
    """
    map vader output e.g.,
    {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.4215}
    to one of the following values:
    a) positive float -> 'positive'
    b) 0.0 -> 'neutral'
    c) negative float -> 'negative'
    
    :param dict vader_output: output dict from vader
    
    :rtype: str
    :return: 'negative' | 'neutral' | 'positive'
    """
    compound = vader_output['compound']
    
    if compound < 0:
        return 'negative'
    elif compound == 0.0:
        return 'neutral'
    elif compound > 0.0:
        return 'positive'
    
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.0}) == 'neutral'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.01}) == 'positive'
assert vader_output_to_label( {'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': -0.01}) == 'negative'

In [24]:
import spacy
import json
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report

nlp = spacy.load('en_core_web_sm')
vader_model = SentimentIntensityAnalyzer()

def run_vader(textual_unit, lemmatize=False, parts_of_speech_to_consider=None, verbose=0):
    doc = nlp(textual_unit)
    input_to_vader = []
    for sent in doc.sents:
        for token in sent:
            to_add = token.lemma_ if lemmatize else token.text
            if to_add == '-PRON-':
                to_add = token.text
            if parts_of_speech_to_consider:
                if token.pos_ in parts_of_speech_to_consider:
                    input_to_vader.append(to_add)
            else:
                input_to_vader.append(to_add)
    return vader_model.polarity_scores(' '.join(input_to_vader))

def vader_output_to_label(vader_output):
    compound = vader_output['compound']
    if compound < 0:
        return 'negative'
    elif compound == 0.0:
        return 'neutral'
    else:
        return 'positive'

tweets = []
all_vader_output = []
gold = []

to_lemmatize = True
pos = set()

for id_, tweet_info in my_tweets.items():
    the_tweet = tweet_info['text_of_tweet']
    vader_output = run_vader(the_tweet, lemmatize=to_lemmatize, parts_of_speech_to_consider=pos)
    vader_label = vader_output_to_label(vader_output)

    tweets.append(the_tweet)
    all_vader_output.append(vader_label)
    gold.append(tweet_info['sentiment_label'])

print(classification_report(gold, all_vader_output))

              precision    recall  f1-score   support

    negative       0.50      0.50      0.50        16
     neutral       0.67      0.35      0.46        17
    positive       0.52      0.76      0.62        17

    accuracy                           0.54        50
   macro avg       0.56      0.54      0.53        50
weighted avg       0.56      0.54      0.53        50



#### Quantitative Evaluation                                                                                                                                                     
                                                                                                                                                                                    
The classification report contains four metrics:                                                                                                                                    
- Precision: it measures of all tweets VADER predicted as a given class, how many were actually that class
- Recall: of all tweets that truly belong to a class, how many did VADER correctly identify?
- F1-score: harmonic mean of precision and recall (it penalises large gaps between the two)
- Support: number of "gold-standard" tweets per class (16 negative, 17 neutral, 17 positive)                                                                                      
                                                                                                                                                                                    
- The macro average F1 (0.53) is the most relevant metric here for us because the classes are roughly balanced, so that means that we care equally about performance across all three. 
- Weighted average gives the same result in this case since support is nearly equal. 
- The overall accuracy is 0.54 which is barely better than random chance for a 3-class problem (chance = 0.33)    

Analysing per class:                                                                                                                                                           
- Positive (F1=0.62) performs best. Recall is high (0.76), meaning VADER catches most positive tweets, but precision is low (0.52), so it over-predicts positive (misclassifying
some neutral tweets as positive)                                                                                                                                                    
- Negative (F1=0.50) is balanced but weak; precision and recall are both exactly 0.50, meaning VADER is uncertain about negative sentiment.
- Neutral (F1=0.46) performs the worst on thsi dataset. Precision is highest (0.67) but recall is very low (0.35); VADER misses most neutral tweets, incorrectly assigning them a positive or negative label. But this is expected since VADER is a sentiment tool designed to detect polarity, so it has a systematic bias toward predicting some sentiment rather than none.             
                                                                                                                                                                        

In [25]:
for tweet, predicted, actual in zip(tweets, all_vader_output, gold):
      if predicted != actual:
          print(f'GOLD: {actual} | PREDICTED: {predicted}')
          print(tweet)
          print()

GOLD: positive | PREDICTED: negative
So Conor McGregor is ALREADY the favorite leading up to the fight in December. I feel the odds will fluctuate quite a bit! #UFC #UFC194

GOLD: positive | PREDICTED: negative
I liked a @user video from @user Mike Tyson: Floyd Mayweather lost the fight in September 12

GOLD: negative | PREDICTED: neutral
"\""""No you may not kick it.\""""  -Tribe Called Quest answering a text from Billy Cundiff"

GOLD: neutral | PREDICTED: positive
"Is2g if I hear one more ""im not racists im just against Islam"" I will lunch myself into the sun"

GOLD: positive | PREDICTED: negative
"SUNDAY, JUNE 19TH is NATIONAL ICE CREAM DAY!!!  (Sorry for the short notice, but our A/C is now back up and...

GOLD: neutral | PREDICTED: positive
"Is there a decent karaoke spot in Tulsa on a Wednesday? I feel the need to do ""Number of the Beast"" by Iron Maiden in honor of @user

GOLD: neutral | PREDICTED: positive
Two more days till the march to the BCS resumes! SDSU gets to experie

#### Positive tweets misclassified as negative:

1. *"So Conor McGregor is ALREADY the favorite leading up to the fight in December. I feel the odds will fluctuate quite a bit! #UFC #UFC194"*
-  The word "fight" has a negative valence in the VADER lexicon. But this tweet expresses fan excitement, and here we can see VADER cannot understand domain context, in sports, "fight" is
neutral/positive.

2. *"I liked a @user video from @user Mike Tyson: Floyd Mayweather lost the fight in September 12"*
- "lost" is negative in the VADER lexicon, but this is a positive tweet (someone sharing a video they liked). VADER fixates on "lost" and ignores the positive framing of "I liked".

3. *"SUNDAY, JUNE 19TH is NATIONAL ICE CREAM DAY!!! (Sorry for the short notice, but our A/C is now back up and..."*
- "Sorry" is scored negatively in VADER, outweighing the excitement of the exclamation marks. The overall tone is cheerful but VADER is misled by a single polite word.


#### Negative tweets misclassified:

4. *"No you may not kick it." —Tribe Called Quest answering a text from Billy Cundiff* → predicted neutral
-  This expresses frustration/rejection but uses no explicitly negative sentiment words. VADER relies on its lexicon and finds nothing to flag, so it defaults to neutral.

5. *"Somewhere in Appleland, Steve Jobs just spit his morning coffee onto his computer screen. EAH!"* → predicted neutral
- The negativity here is implicit and requires cultural knowledge (Apple mocking a competitor here). No negative lexicon words are present, so VADER cannot detect it.


#### Neutral tweets misclassified:

6. *"Is2g if I hear one more 'im not racists im just against Islam' I will lunch myself into the sun"* → predicted positive
- Sarcastic frustration. VADER picks up no negative lexicon words and may score "sun" slightly positive, missing the sarcasm entirely.

7. *"Is there a decent karaoke spot in Tulsa on a Wednesday? I feel the need to do 'Number of the Beast'"* → predicted positive
- "feel the need" and "decent" have slightly positive scores in the lexicon. The tweet is a neutral question but VADER over-interprets these words.

8. *"Two more days till the march to the BCS resumes! SDSU gets to experience what a top ranked defense looks like up close. #GoBroncos"* → predicted positive
- "top ranked" scores positively in VADER. The tweet is factual sports news, but superlatives confuse VADER into assigning positive sentiment.

9. *"Who is the 6'4 guard from Yelm that dropped 44 in the Curtis jamboree Monday night?"* → predicted negative
- "dropped" likely has a negative score in VADER (to drop = to lose/fail). Here it means scoring 44 points, which is seen as impressive (so positive), thsi is a classic word sense ambiguity.


So what we can see for error patterns here is:
- Domain-specific vocabulary (fight, dropped, lost) has different meaning in sports context than in VADER's general lexicon (first tweet example)
- Sarcasm and irony seem completely invisible to VADER
- implicit sentiment requiring world knowledge cannot be captured by lexicon lookup
- Polite/formulaic words (like "sorry") can override the overall tone

### [4 points] Question 4:
Run VADER on the set of airline tweets with the following settings:

* Run VADER (as it is) on the set of airline tweets 
* Run VADER on the set of airline tweets after having lemmatized the text
* Run VADER on the set of airline tweets with only adjectives
* Run VADER on the set of airline tweets with only adjectives and after having lemmatized the text
* Run VADER on the set of airline tweets with only nouns
* Run VADER on the set of airline tweets with only nouns and after having lemmatized the text
* Run VADER on the set of airline tweets with only verbs
* Run VADER on the set of airline tweets with only verbs and after having lemmatized the text

* [1 point] a. Generate for all separate experiments the classification report, i.e., Precision, Recall, and F<sub>1</sub> scores per category as well as micro and macro averages. **Use a different code cell (or multiple code cells) for each experiment.**
* [3 points] b. Compare the scores and explain what they tell you.
* - Does lemmatisation help? Explain why or why not.
* - Are all parts of speech equally important for sentiment analysis? Explain why or why not.

In [26]:
import pathlib
import zipfile
from sklearn.datasets import load_files
from sklearn.metrics import classification_report

dataset_dir = pathlib.Path('airlinetweets')
if not dataset_dir.exists():
    with zipfile.ZipFile('airlinetweets.zip', 'r') as z:
        z.extractall('.')

airline_data = load_files(str(dataset_dir), encoding='utf-8', decode_error='replace')
airline_tweets = airline_data.data
airline_labels = [airline_data.target_names[t] for t in airline_data.target]

print(f'Loaded {len(airline_tweets)} tweets')


Loaded 4755 tweets


In [27]:
def run_experiment(tweets, labels, lemmatize=False, pos=None, name=''):
      predictions = [vader_output_to_label(run_vader(t, lemmatize=lemmatize, parts_of_speech_to_consider=pos)) for t in tweets]
      print(f'\n=== {name} ===')
      print(classification_report(labels, predictions))

In [28]:
run_experiment(airline_tweets, airline_labels, name='VADER default')


=== VADER default ===
              precision    recall  f1-score   support

    negative       0.80      0.51      0.63      1750
     neutral       0.60      0.51      0.55      1515
    positive       0.56      0.88      0.68      1490

    accuracy                           0.63      4755
   macro avg       0.65      0.64      0.62      4755
weighted avg       0.66      0.63      0.62      4755



In [29]:
run_experiment(airline_tweets, airline_labels, lemmatize=True, name='VADER lemmatized')


=== VADER lemmatized ===
              precision    recall  f1-score   support

    negative       0.78      0.52      0.63      1750
     neutral       0.60      0.49      0.54      1515
    positive       0.56      0.88      0.68      1490

    accuracy                           0.62      4755
   macro avg       0.65      0.63      0.62      4755
weighted avg       0.65      0.62      0.62      4755



In [30]:
run_experiment(airline_tweets, airline_labels, pos={'ADJ'}, name='VADER adjectives only')


=== VADER adjectives only ===
              precision    recall  f1-score   support

    negative       0.86      0.20      0.33      1750
     neutral       0.40      0.89      0.55      1515
    positive       0.67      0.44      0.53      1490

    accuracy                           0.50      4755
   macro avg       0.64      0.51      0.47      4755
weighted avg       0.65      0.50      0.46      4755



In [31]:
run_experiment(airline_tweets, airline_labels, lemmatize=True, pos={'ADJ'}, name='VADER adjectives + lemmatized')


=== VADER adjectives + lemmatized ===
              precision    recall  f1-score   support

    negative       0.86      0.20      0.33      1750
     neutral       0.40      0.89      0.55      1515
    positive       0.67      0.44      0.53      1490

    accuracy                           0.50      4755
   macro avg       0.64      0.51      0.47      4755
weighted avg       0.65      0.50      0.46      4755



In [32]:
run_experiment(airline_tweets, airline_labels, pos={'NOUN'}, name='VADER nouns only')


=== VADER nouns only ===
              precision    recall  f1-score   support

    negative       0.73      0.14      0.23      1750
     neutral       0.36      0.82      0.50      1515
    positive       0.53      0.35      0.42      1490

    accuracy                           0.42      4755
   macro avg       0.54      0.44      0.39      4755
weighted avg       0.55      0.42      0.38      4755



In [33]:
run_experiment(airline_tweets, airline_labels, lemmatize=True, pos={'NOUN'}, name='VADER nouns + lemmatized')


=== VADER nouns + lemmatized ===
              precision    recall  f1-score   support

    negative       0.71      0.15      0.25      1750
     neutral       0.36      0.81      0.50      1515
    positive       0.52      0.34      0.41      1490

    accuracy                           0.42      4755
   macro avg       0.53      0.44      0.39      4755
weighted avg       0.54      0.42      0.38      4755



In [34]:
run_experiment(airline_tweets, airline_labels, pos={'VERB'}, name='VADER verbs only')


=== VADER verbs only ===
              precision    recall  f1-score   support

    negative       0.79      0.29      0.42      1750
     neutral       0.38      0.81      0.52      1515
    positive       0.57      0.35      0.43      1490

    accuracy                           0.47      4755
   macro avg       0.58      0.48      0.46      4755
weighted avg       0.59      0.47      0.46      4755



In [35]:
run_experiment(airline_tweets, airline_labels, lemmatize=True, pos={'VERB'}, name='VADER verbs + lemmatized')


=== VADER verbs + lemmatized ===
              precision    recall  f1-score   support

    negative       0.75      0.29      0.42      1750
     neutral       0.38      0.78      0.51      1515
    positive       0.58      0.36      0.44      1490

    accuracy                           0.47      4755
   macro avg       0.57      0.48      0.46      4755
weighted avg       0.58      0.47      0.45      4755



To help us compare we made a table summarizing the results;
| Experiment                  | Macro F1 |
|-----------------------------|----------|
| VADER default               | 0.62     |
| VADER lemmatized            | 0.62     |
| VADER adjectives only       | 0.47     |
| VADER adjectives + lemmatized | 0.47   |
| VADER nouns only            | 0.39     |
| VADER nouns + lemmatized    | 0.39     |
| VADER verbs only            | 0.46     |
| VADER verbs + lemmatized    | 0.46     |

So does lemmatisation help?
On this dataset, it seems like no, here lemmatisation makes no meaningful difference. Every experiment with lemmatisation produces identical or marginally lower scores than its non-lemmatised counterpart (e.g. default: 0.62 vs lemmatized: 0.62, adjectives: 0.47 vs adjectives+lemmatized: 0.47). This is expected because VADER's lexicon contains wordforms rather than lemmas. Words like "loved", "loves", and "love" are all likely already covered as separate entries in the lexicon, so reducing them to their base form does not add new matches. In fact, lemmatisation can occasionally hurt because spaCy's lemmatiser may produce a form not present in VADER's lexicon.

And are all parts of speech equally important for sentiment analysis?

Again from the experiments on thsi dataset it seems like nthe answer to thsi question is no: the results clearly show that not all POS contribute equally:
- Full text (default) performs best (macro F1 = 0.62): using all words gives VADER the most information and produces the highest scores across all classes
- Adjectives only (F1 = 0.47) performs second best among the POS-filtered experiments. Adjectives are the most sentiment-bearing part of speech ("terrible", "great",
"delayed") and are widely used in VADER's lexicon. However, many airline tweets express sentiment through nouns and verbs ("delay", "cancelled", "love"), so filtering to
adjectives alone loses important signal.
- Verbs only (F1 = 0.46) performs similarly to adjectives. Verbs like "cancelled", "lost", "love", "hate" carry sentiment in this domain, but are less reliable than adjectives
overall. Recall for negative tweets drops sharply (0.29) since negative airline sentiment is often expressed through nouns and adjectives rather than verbs.
- Nouns only (F1 = 0.39) performs worst. Nouns are generally the least sentiment-bearing POS — they describe entities and events rather than opinions. The airline domain is a
partial exception ("delay", "nightmare", "disaster") but these are far outnumbered by neutral nouns ("flight", "seat", "gate"), causing VADER to default to neutral for most tweets
(neutral recall = 0.82).

So given these experiments, in conclusion, full text is always best for VADER. Among individual POS, adjectives are most useful for sentiment, followed by verbs, with nouns contributing the least. This seems to align with linguistic intuition: opinions are most directly expressed through evaluative adjectives.


## Part II: scikit-learn assignments
### [4 points] Question 5
Train the scikit-learn classifier (Naive Bayes) using the airline tweets.

+ Train the model on the airline tweets with 80% training and 20% test set and default settings (TF-IDF representation, min_df=2)
+ Train with different settings:
    + with respect to vectorizing: TF-IDF ('airline_tfidf') vs. Bag of words representation ('airline_count') 
    + with respect to the frequency threshold (min_df). Carry out experiments with increasing values for document frequency (min_df = 2; min_df = 5; min_df =10) 
* [1 point] a. Generate a classification_report for all experiments
* [3 points] b. Look at the results of the experiments with the different settings and try to explain why they differ: 
    + which category performs best, is this the case for any setting?
    + does the frequency threshold affect the scores? Why or why not according to you?

In [40]:
# Q5a - Naive Bayes experiments on airline tweets
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report


# 1) Load data
lab3_dir = pathlib.Path.cwd()
zip_path = lab3_dir / "airlinetweets.zip"
data_dir = lab3_dir / "airlinetweets"

if not data_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(lab3_dir)

airline_data = load_files(str(data_dir), encoding="utf-8", decode_error="replace")
texts = airline_data.data
y = airline_data.target
target_names = airline_data.target_names


def run_nb_experiment(texts, y, vectorizer_kind="tfidf", min_df=2, test_size=0.2, random_state=42):
    X_train_text, X_test_text, y_train, y_test = train_test_split(
        texts, y, test_size=test_size, random_state=random_state, stratify=y
    )

    if vectorizer_kind == "tfidf":
        vectorizer = TfidfVectorizer(min_df=min_df)
    elif vectorizer_kind == "count":
        vectorizer = CountVectorizer(min_df=min_df)
    else:
        raise ValueError("vectorizer_kind must be 'tfidf' or 'count'")

    X_train = vectorizer.fit_transform(X_train_text)
    X_test = vectorizer.transform(X_test_text)

    clf = MultinomialNB()
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    report_dict = classification_report(
        y_test, y_pred, target_names=target_names, output_dict=True, zero_division=0
    )

    print(f"\n=== {vectorizer_kind.upper()} | min_df={min_df} ===")
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))

    return {
        "vectorizer": vectorizer_kind,
        "min_df": min_df,
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    }


# 3) Required experiments
settings = [
    ("tfidf", 2),
    ("tfidf", 5),
    ("tfidf", 10),
    ("count", 2),
    ("count", 5),
    ("count", 10),
]

results = []
for vec_kind, min_df in settings:
    row = run_nb_experiment(texts, y, vectorizer_kind=vec_kind, min_df=min_df)
    results.append(row)


results_df = pd.DataFrame(results).sort_values(by="macro_f1", ascending=False)
print("\nSummary (sorted by macro_f1):")
print(results_df.to_string(index=False))


=== TFIDF | min_df=2 ===
              precision    recall  f1-score   support

    negative       0.80      0.94      0.86       350
     neutral       0.82      0.69      0.75       303
    positive       0.83      0.80      0.82       298

    accuracy                           0.81       951
   macro avg       0.82      0.81      0.81       951
weighted avg       0.82      0.81      0.81       951


=== TFIDF | min_df=5 ===
              precision    recall  f1-score   support

    negative       0.81      0.92      0.86       350
     neutral       0.81      0.72      0.76       303
    positive       0.83      0.80      0.81       298

    accuracy                           0.82       951
   macro avg       0.82      0.81      0.81       951
weighted avg       0.82      0.82      0.81       951


=== TFIDF | min_df=10 ===
              precision    recall  f1-score   support

    negative       0.82      0.91      0.86       350
     neutral       0.79      0.72      0.76       

Across all settings, the negative class performs best (F1 between 0.86–0.89), while neutral is consistently the hardest class (F1 around 0.75–0.76). This likely happens because negative airline tweets contain sentiments words which are easy to classify as conveying a negative sentiment (e.g., complaints, delays, cancellations), whereas neutral tweets are linguistically less distinctive as neutral sentiment is difficult to detect, involving the lack of negative or positive sentiment rather than a having easily definable or identifiable of its own.

The best overall model is Bag of Words (CountVectorizer) with min_df=2, which gives the highest macro-F1 (0.819) and accuracy (0.825). In general, Count-based models outperform TF-IDF in these experiments. I think that is because Multinomial Naive Bayes benefits from raw token frequency counts, while TF-IDF downweights frequent words that may still be useful sentiment cues in this domain.

Increasing the frequency threshold (min_df) has a small but consistent negative effect on overall performance. For Count, macro-F1 drops from 0.819 (min_df=2) to 0.818 (min_df=5) to 0.815 (min_df=10); TF-IDF shows a similar pattern overall. This suggests that removing rarer words reduces vocabulary noise only slightly, but also removes informative low-frequency sentiment terms, especially useful for class distinctions.

Overall, representation and min_df do affect results, but differences are moderate; the most robust setting here is Count + min_df=2, with strongest performance on negative tweets and persistent difficulty on neutral tweets.

### [4 points] Question 6: Inspecting the best scoring features

+ Train the scikit-learn classifier (Naive Bayes) model with the following settings (airline tweets 80% training and 20% test;  Bag of words representation ('airline_count'), min_df=2)
* [1 point] a. Generate the list of best scoring features per class (see function **important_features_per_class** below) [1 point]
* [3 points] b. Look at the lists and consider the following issues:
    + [1 point] Which features did you expect for each separate class and why?
    + [1 point] Which features did you not expect and why ?
    + [1 point] The list contains all kinds of words such as names of airlines, punctuation, numbers and content words (e.g., 'delay' and 'bad'). Which words would you remove or keep when trying to improve the model and why?

In [44]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

X_train_text, X_test_text, y_train, y_test = train_test_split(
    airline_tweets,
    airline_data.target,
    test_size=0.2,
    random_state=42,
    stratify=airline_data.target
)

airline_count = CountVectorizer(min_df=2)
X_train_count = airline_count.fit_transform(X_train_text)
X_test_count = airline_count.transform(X_test_text)

clf = MultinomialNB()
clf.fit(X_train_count, y_train)

y_pred = clf.predict(X_test_count)
print(classification_report(y_test, y_pred, target_names=airline_data.target_names))

def important_features_per_class(vectorizer,classifier,n=80):
    class_labels = classifier.classes_
    feature_names = vectorizer.get_feature_names_out()
    topn_class1 = sorted(zip(classifier.feature_count_[0], feature_names),reverse=True)[:n]
    topn_class2 = sorted(zip(classifier.feature_count_[1], feature_names),reverse=True)[:n]
    topn_class3 = sorted(zip(classifier.feature_count_[2], feature_names),reverse=True)[:n]
    print("Important words in negative documents")
    for coef, feat in topn_class1:
        print(class_labels[0], coef, feat)
    print("-----------------------------------------")
    print("Important words in neutral documents")
    for coef, feat in topn_class2:
        print(class_labels[1], coef, feat) 
    print("-----------------------------------------")
    print("Important words in positive documents")
    for coef, feat in topn_class3:
        print(class_labels[2], coef, feat) 



important_features_per_class(airline_count, clf, n=30)

# example of how to call from notebook:
#important_features_per_class(airline_vec, clf)

              precision    recall  f1-score   support

    negative       0.84      0.94      0.89       350
     neutral       0.83      0.71      0.76       303
    positive       0.81      0.82      0.81       298

    accuracy                           0.83       951
   macro avg       0.82      0.82      0.82       951
weighted avg       0.82      0.83      0.82       951

Important words in negative documents
0 1390.0 united
0 756.0 to
0 511.0 the
0 385.0 you
0 382.0 flight
0 368.0 and
0 335.0 for
0 326.0 on
0 319.0 my
0 275.0 is
0 270.0 in
0 209.0 of
0 203.0 your
0 197.0 that
0 196.0 it
0 170.0 me
0 163.0 not
0 161.0 have
0 152.0 at
0 150.0 no
0 148.0 with
0 143.0 was
0 119.0 this
0 118.0 service
0 117.0 we
0 117.0 can
0 114.0 be
0 106.0 an
0 102.0 virginamerica
0 100.0 from
-----------------------------------------
Important words in neutral documents
1 609.0 to
1 315.0 jetblue
1 274.0 the
1 267.0 southwestair
1 258.0 united
1 251.0 on
1 240.0 you
1 231.0 flight
1 202.0 for
1 1

For the negative class I expected obviously negative words like: terrible, horrible, etc. Also many mentions of the words not and no. These words seem obvious it is very uncommon to use the negative words with negations e.g. "not terrible" and they would be easily classifiable.

For the neutral class I did not know what to expect, I don't think there are many words that are characteristic of neutral sentiment so I expected most common words to verbs and nouns.

For the positive class I expected words for thanking and for positive descriptions like: thanks, good, love, etc. These are obvious as they are very likely to convey a positive sentiment

For all the classes I was not expecting to see so many words of "common use" like: on, for, the, you, it, me, at, etc. Also I was not expecting to see particular airline names.

I would definitely keep the name of the airlines, as they can serve as a way to study the perception of good an airline is. I would remove most of the "common use" words that were named before as they clearly don't belong to any kind of sentiment, instead are just words common to the expression of sentiment.



### [Optional! (will not  be graded)] Question 7
Train the model on airline tweets and test it on your own set of tweets
+ Train the model with the following settings (airline tweets 80% training and 20% test;  Bag of words representation ('airline_count'), min_df=2)
+ Apply the model on your own set of tweets and generate the classification report
* [1 point] a. Carry out a quantitative analysis.
* [1 point] b. Carry out an error analysis on 10 correctly and 10 incorrectly classified tweets and discuss them
* [2 points] c. Compare the results (cf. classification report) with the results obtained by VADER on the same tweets and discuss the differences.

### [Optional! (will not be graded)] Question 8: trying to improve the model
* [2 points] a. Think of some ways to improve the scikit-learn Naive Bayes model by playing with the settings or applying linguistic preprocessing (e.g., by filtering on part-of-speech, or removing punctuation). Do not change the classifier but continue using the Naive Bayes classifier. Explain what the effects might be of these other settings 
+ [1 point] b. Apply the model with at least one new setting (train on the airline tweets using 80% training, 20% test) and generate the scores
* [1 point] c. Discuss whether the model achieved what you expected.

## End of this notebook